# Silver_OneSite_LAH_DF2

Lee archivos XLSX procesados desde **Bronze Lakehouse** y escribe tablas Delta limpias a **Silver Lakehouse**.

| Step | Tabla Silver | Fuente |
|---|---|---|
| 4 | `os_ar_collections` | ar_collections.xlsx |
| 5 | `os_denies_cancells` | denies / cancells.xlsx |
| 6 | `os_effective_rents` | effective_rents.xlsx |
| 7 | `os_move_ins` | move_ins.xlsx |
| 8 | `os_delinquent_prepaid` | delinquent_prepaid.xlsx |
| 9 | `os_leasing_desk` | leasing_desk.xlsx |
| 10 | Validación | row counts, 9 propiedades |

## Config

In [ ]:
# ============================================================
# Silver_OneSite_LAH_DF2 — CONFIG
# ============================================================
# HOW TO USE:
#   1. Attach THIS notebook to Heritage_Silver_LH (default lakehouse)
#   2. Also add Heritage_Bronze_LH as a second lakehouse
#   3. Set BRONZE_LH_NAME to the exact Fabric lakehouse display name
# ============================================================

BRONZE_LH_NAME = "Heritage_Bronze_LH"
SHEETS_DIR     = "Files/xlsx_by_sheet"   # inside Bronze LH

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, IntegerType
import traceback

print(f"Bronze LH : {BRONZE_LH_NAME}")
print(f"Sheets dir: {SHEETS_DIR}")

## Shared Helpers

In [ ]:
# ============================================================
# SHARED HELPERS
# ============================================================
from notebookutils import mssparkutils

def _ls_bronze(subdir=""):
    path = SHEETS_DIR + ("/" + subdir if subdir else "")
    try:    return mssparkutils.fs.ls(path)
    except: return []

def _find_file(keyword, subdir=""):
    kw = keyword.lower()
    for it in _ls_bronze(subdir):
        nm = (getattr(it,"name",None) or it.path.split("/")[-1]).lower()
        if kw in nm:
            return getattr(it,"path", f"{SHEETS_DIR}/{nm}")
    return None

def _read_xlsx(path, header_row=0):
    import pandas as pd
    local = "/tmp/os_tmp.xlsx"
    mssparkutils.fs.cp(path, f"file:{local}", True)
    df = pd.read_excel(local, header=header_row)
    return spark.createDataFrame(df.astype(str).where(df.notna(), None))

def _safe_double(col_name):
    return F.when(
        F.trim(F.col(col_name)).rlike(r'^-?[0-9]+(\.[0-9]+)?$'),
        F.col(col_name).cast(DoubleType())
    ).otherwise(F.lit(None).cast(DoubleType()))

def _save(df, tbl, overwrite=True):
    mode = "overwrite" if overwrite else "append"
    df.write.format("delta").mode(mode).option("overwriteSchema","true").saveAsTable(tbl)
    cnt = spark.table(tbl).count()
    print(f"  Saved: {tbl} | rows={cnt}")
    return cnt

print("Helpers loaded.")

## Step 4 — os_ar_collections

In [ ]:
# ============================================================
# STEP 4 — ar_collections
# Source: Bronze LH / xlsx_by_sheet / ar_collections.xlsx (or Delinquent sheet)
# Schema: Property | Current | 30-59 Days | 60-89 Days | 90+ Days | Total
# ============================================================
def step4_ar_collections():
    path = _find_file("ar_collections") or _find_file("delinquent_prepaid")
    if not path:
        raise FileNotFoundError("ar_collections file not found in Bronze LH")
    print(f"  Reading: {path}")

    import pandas as pd
    from io import BytesIO

    local = "/tmp/ar_collections.xlsx"
    mssparkutils.fs.cp(path, f"file:{local}", True)

    def is_2130(v):
        try:    return abs(float(str(v).strip()) - 2130.0) < 0.01
        except: return False

    rows = []
    with open(local, "rb") as fh:
        wb = __import__("openpyxl").load_workbook(BytesIO(fh.read()), data_only=True)
    ws = wb.active
    header_row = None
    data_rows  = []
    for r in ws.iter_rows(values_only=True):
        row = [str(c).strip() if c is not None else "" for c in r]
        if header_row is None:
            if any(k in " ".join(row).lower() for k in ["property","current","30","60","90"]):
                header_row = row
        else:
            data_rows.append(row)

    if not header_row or not data_rows:
        raise ValueError("Could not find header row in ar_collections file")

    def safe_float(v):
        try:    return float(str(v).strip().replace(",","").replace("$",""))
        except: return None

    for row in data_rows:
        if len(row) < 5: continue
        prop = row[0]
        if not prop or prop.lower() in ("","total","subtotal","property","none"): continue

        # Detect which columns are account / amount pairs (like Power Query M logic)
        # Skip rows that are only account code / label rows
        # Sum non-2130 values into Current vs 30+ Days buckets
        col1_vals = [(row[i] if i < len(row) else "") for i in range(len(row))]

        # Simple flat layout: Property | Current | 30-59 | 60-89 | 90+ | Total
        try:
            current  = safe_float(row[1]) or 0.0
            d30_59   = safe_float(row[2]) or 0.0
            d60_89   = safe_float(row[3]) or 0.0
            d90plus  = safe_float(row[4]) or 0.0
            total    = safe_float(row[5]) if len(row) > 5 else (current + d30_59 + d60_89 + d90plus)
            rows.append({
                "Property": prop,
                "Current":  current,
                "Days_30_59": d30_59,
                "Days_60_89": d60_89,
                "Days_90_plus": d90plus,
                "Total":  total or 0.0
            })
        except Exception:
            continue

    if not rows:
        raise ValueError("No valid data rows found in ar_collections")

    schema = ["Property","Current","Days_30_59","Days_60_89","Days_90_plus","Total"]
    import pandas as pd
    pdf = pd.DataFrame(rows, columns=schema)
    df  = spark.createDataFrame(pdf)
    df  = df.withColumn("_load_date", F.current_date())
    _save(df, "os_ar_collections")
    df.show(10, truncate=False)

## Step 5 — os_denies_cancells

In [ ]:
# ============================================================
# STEP 5 — denies_cancells
# Source: xlsx_by_sheet / denies*.xlsx or cancell*.xlsx
# ============================================================
def step5_denies():
    path = _find_file("denie") or _find_file("cancell") or _find_file("denial")
    if not path:
        raise FileNotFoundError("denies/cancells file not found in Bronze LH")
    print(f"  Reading: {path}")
    df = _read_xlsx(path)
    df = df.withColumn("_load_date", F.current_date())

    # Standardize column names to snake_case
    renamed = {c: c.lower().strip().replace(" ","_").replace("/","_").replace("-","_") for c in df.columns}
    for old, new in renamed.items():
        if old != new:
            df = df.withColumnRenamed(old, new)

    _save(df, "os_denies_cancells")
    df.show(5, truncate=False)

## Step 6 — os_effective_rents

In [ ]:
# ============================================================
# STEP 6 — effective_rents
# Source: xlsx_by_sheet / effective*.xlsx
# ============================================================
def step6_effective_rents():
    path = _find_file("effective")
    if not path:
        raise FileNotFoundError("effective_rents file not found in Bronze LH")
    print(f"  Reading: {path}")
    df = _read_xlsx(path)
    df = df.withColumn("_load_date", F.current_date())

    renamed = {c: c.lower().strip().replace(" ","_").replace("/","_").replace("-","_") for c in df.columns}
    for old, new in renamed.items():
        if old != new:
            df = df.withColumnRenamed(old, new)

    _save(df, "os_effective_rents")
    df.show(5, truncate=False)

## Step 7 — os_move_ins

In [ ]:
# ============================================================
# STEP 7 — move_ins
# Source: xlsx_by_sheet / move*.xlsx or movein*.xlsx
# ============================================================
def step7_move_ins():
    path = _find_file("move_in") or _find_file("movein") or _find_file("move in")
    if not path:
        raise FileNotFoundError("move_ins file not found in Bronze LH")
    print(f"  Reading: {path}")
    df = _read_xlsx(path)
    df = df.withColumn("_load_date", F.current_date())

    renamed = {c: c.lower().strip().replace(" ","_").replace("/","_").replace("-","_") for c in df.columns}
    for old, new in renamed.items():
        if old != new:
            df = df.withColumnRenamed(old, new)

    _save(df, "os_move_ins")
    df.show(5, truncate=False)

## Step 8 — os_delinquent_prepaid

In [ ]:
# ============================================================
# STEP 8 — delinquent_prepaid
# Source: xlsx_by_sheet / delinquent*.xlsx
# Logic: Same as ar_collections but full detail (account code level)
#        Excludes 2130.xxx payment codes from balance
# ============================================================
def step8_delinquent_prepaid():
    path = _find_file("delinquent")
    if not path:
        raise FileNotFoundError("delinquent_prepaid file not found in Bronze LH")
    print(f"  Reading: {path}")
    df = _read_xlsx(path)
    df = df.withColumn("_load_date", F.current_date())

    renamed = {c: c.lower().strip().replace(" ","_").replace("/","_").replace("-","_") for c in df.columns}
    for old, new in renamed.items():
        if old != new:
            df = df.withColumnRenamed(old, new)

    # Flag 2130.xxx payment code rows
    acct_col = next((c for c in df.columns if "account" in c.lower() or "code" in c.lower()), None)
    if acct_col:
        df = df.withColumn("_is_payment_code",
            F.when(
                F.abs(F.col(acct_col).cast(DoubleType()) - F.lit(2130.0)) < F.lit(0.01),
                F.lit(True)
            ).otherwise(F.lit(False))
        )

    _save(df, "os_delinquent_prepaid")
    df.show(5, truncate=False)

## Step 9 — os_leasing_desk

In [ ]:
# ============================================================
# STEP 9 — leasing_desk
# Source: xlsx_by_sheet / leasing*.xlsx
# ============================================================
def step9_leasing_desk():
    path = _find_file("leasing")
    if not path:
        raise FileNotFoundError("leasing_desk file not found in Bronze LH")
    print(f"  Reading: {path}")
    df = _read_xlsx(path)
    df = df.withColumn("_load_date", F.current_date())

    renamed = {c: c.lower().strip().replace(" ","_").replace("/","_").replace("-","_") for c in df.columns}
    for old, new in renamed.items():
        if old != new:
            df = df.withColumnRenamed(old, new)

    _save(df, "os_leasing_desk")
    df.show(5, truncate=False)

## Step 10 — Data Validation

In [ ]:
# ============================================================
# STEP 10 — DATA VALIDATION
# ============================================================
SILVER_OS_TABLES = [
    "os_ar_collections",
    "os_denies_cancells",
    "os_effective_rents",
    "os_move_ins",
    "os_delinquent_prepaid",
    "os_leasing_desk",
]

def step10_validate():
    checks = []
    print("=== Row counts ===")
    for tbl in SILVER_OS_TABLES:
        try:
            cnt = spark.table(tbl).count()
            ok  = cnt > 0
            checks.append({"check": tbl, "count": cnt, "status": "PASS" if ok else "FAIL"})
            print(f"  {'OK' if ok else 'FAIL'} {tbl}: {cnt:,} rows")
        except Exception as e:
            checks.append({"check": tbl, "count": -1, "status": "FAIL"})
            print(f"  FAIL {tbl}: {e}")

    print("\n=== ar_collections: 9 properties, no negative balances ===")
    try:
        ar = spark.table("os_ar_collections")
        prop_cnt  = ar.filter(F.col("Property").isNotNull()).select("Property").distinct().count()
        neg_curr  = ar.filter(F.col("Current").cast(DoubleType()) < 0).count()
        ok_props  = prop_cnt >= 9
        ok_neg    = neg_curr == 0
        checks.append({"check":"ar properties>=9",  "count":prop_cnt, "status":"PASS" if ok_props else "FAIL"})
        checks.append({"check":"ar no neg Current",  "count":neg_curr, "status":"PASS" if ok_neg   else "FAIL"})
        print(f"  {'OK' if ok_props else 'FAIL'} Properties: {prop_cnt}")
        print(f"  {'OK' if ok_neg else 'FAIL'} Negative Current balances: {neg_curr}")
    except Exception as e:
        print(f"  FAIL ar_collections validation: {e}")

    failed = [c for c in checks if c["status"]=="FAIL"]
    passed = [c for c in checks if c["status"]=="PASS"]
    print(f"\nValidation Summary: {len(passed)} PASS | {len(failed)} FAIL")

    if failed:
        for f in failed:
            print(f"  FAIL: {f['check']} (count={f['count']})")
        raise Exception(f"OneSite Silver validation: {len(failed)} checks failed.")

    print("All OneSite Silver validations passed.")

## Pipeline Runner

In [ ]:
# ============================================================
# PIPELINE RUNNER — Silver OneSite
# ============================================================
import traceback

_RESULTS = []

def _run(name, fn):
    print(f"\n{'='*55}")
    print(f"START: {name}")
    print("="*55)
    try:
        fn()
        _RESULTS.append((name,"OK"))
        print(f"DONE: {name}")
    except Exception as e:
        _RESULTS.append((name, str(e)[:300]))
        print(f"FAIL: {name}: {e}")
        traceback.print_exc()

_run("Step 4  — ar_collections",         step4_ar_collections)
_run("Step 5  — denies_cancells",         step5_denies)
_run("Step 6  — effective_rents",         step6_effective_rents)
_run("Step 7  — move_ins",                step7_move_ins)
_run("Step 8  — delinquent_prepaid",      step8_delinquent_prepaid)
_run("Step 9  — leasing_desk",            step9_leasing_desk)
_run("Step 10 — Validation",              step10_validate)

print("\n" + "="*55)
print("SILVER ONESITE PIPELINE SUMMARY")
print("="*55)
for name, status in _RESULTS:
    icon = "OK" if status == "OK" else "FAIL"
    print(f"  [{icon}] {name}" + (f": {status}" if status != "OK" else ""))

if any(s != "OK" for _,s in _RESULTS):
    raise Exception("Silver OneSite pipeline had failures.")
print("\nSilver OneSite complete.")